# Q-Neural-Dynamics: Quantum-Inspired Neural Framework
### CPU-Optimized Execution Notebook for Biological & Neural Time-Series Analysis
**Author & Lead Researcher:** Amir Kabirian

In [ ]:
# Step 0: Environment Setup & Library Imports (Pure PyTorch & NumPy - Zero External Conflicts)
# Setting up the workspace and enforcing strict CPU device allocation to prevent hardware conflicts.
print("Setting up clean environment...")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Enforcing strict CPU device allocation
device = torch.device(
    "cpu"
)  # Explicitly targeting CPU runtime for stable execution
print(f"Active computational device: {device}")

In [ ]:
# Step 1: Define Quantum-Inspired Orthogonal Reservoir Layer & Hybrid Classifier
# Implements unitary-inspired orthogonal recurrent weights via QR decomposition for stable temporal dynamics.
class QuantumInspiredReservoir(nn.Module):
    def __init__(self, input_dim, reservoir_dim):
        super(QuantumInspiredReservoir, self).__init__()
        self.input_dim = input_dim
        self.reservoir_dim = reservoir_dim

        # QR decomposition for orthogonal transformation (unitary-inspired weights)
        raw_weights = torch.randn(reservoir_dim, input_dim)
        Q, _ = torch.linalg.qr(raw_weights)
        self.W_in = nn.Parameter(Q, requires_grad=False)  # Frozen input projection weights

        recurrent_raw = torch.randn(reservoir_dim, reservoir_dim)
        Q_rec, _ = torch.linalg.qr(recurrent_raw)
        self.W_rec = nn.Parameter(Q_rec * 0.99, requires_grad=False)  # Scaled orthogonal recurrent weights

    def forward(self, x):
        # Processing time-series sequences step-by-step through the quantum-inspired reservoir
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.reservoir_dim, device=x.device)

        states = []
        for t in range(seq_len):
            xt = x[:, t, :]
            # State update equation using tanh non-linearity and matrix multiplications
            h = torch.tanh(torch.matmul(xt, self.W_in.T) + torch.matmul(h, self.W_rec.T))
            states.append(h.unsqueeze(1))

        return torch.cat(states, dim=1)

class QNeuralClassifier(nn.Module):
    def __init__(self, input_dim, reservoir_dim, num_classes):
        super(QNeuralClassifier, self).__init__()
        self.reservoir = QuantumInspiredReservoir(input_dim, reservoir_dim)
        # Fully connected feed-forward classifier network mapping reservoir features to classes
        self.classifier = nn.Sequential(
            nn.Linear(reservoir_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Extracting temporal reservoir representations and pooling across time
        res_states = self.reservoir(x)
        repr_features = torch.mean(res_states, dim=1)  # Temporal average pooling
        out = self.classifier(repr_features)
        return out

print("Model architecture initialized successfully on CPU.")

In [ ]:
# Step 2: Dataset Loading & Native Preprocessing (Strict Binary Labels 0 or 1)
# Handles data standardization and sliding-window sequence creation natively via NumPy.
def load_and_preprocess_data(file_path, target_column, seq_len=30):
    df = pd.read_csv(file_path)
    y = df[target_column].values
    X = df.drop(columns=[target_column]).values

    # Native NumPy Standardization (Zero-mean, unit-variance calculation)
    mean = X.mean(axis=0)
    std = X.std(axis=0) + 1e-8
    X_scaled = (X - mean) / std

    # Constructing sliding time-windows for sequence classification
    num_samples = len(X_scaled) - seq_len
    X_sequences, y_sequences = [], []
    for i in range(num_samples):
        X_sequences.append(X_scaled[i : i + seq_len])
        y_sequences.append(y[i + seq_len])

    return np.array(X_sequences, dtype=np.float32), np.array(y_sequences, dtype=np.int64)

# Generate biological benchmark dataset with strict binary labels (0 or 1) for pipeline verification
np.random.seed(42)
features = np.random.randn(1000, 10)
labels = np.random.randint(0, 2, size=(1000, 1))
sample_data = np.hstack((features, labels))
cols = [f"feature_{i}" for i in range(10)] + ["label"]
pd.DataFrame(sample_data, columns=cols).to_csv("biological_data.csv", index=False)
print("Benchmark dataset with binary labels generated successfully.")

In [ ]:
# Step 3: Training Pipeline, Loss Tracking & Advanced Evaluation Metrics
# Executes the model training loop, computes Cross-Entropy loss, and evaluates clinical metrics (F1, Precision, Recall).
X_data, y_data = load_and_preprocess_data('biological_data.csv', 'label', seq_len=20)

# Native split (80% training data, 20% testing data)
split_idx = int(len(X_data) * 0.8)
X_train, X_test = X_data[:split_idx], X_data[split_idx:]
y_train, y_test = y_data[:split_idx], y_data[split_idx:]

# Transferring processed arrays into PyTorch tensors on CPU
X_train_t = torch.tensor(X_train, device=device)
y_train_t = torch.tensor(y_train, device=device)
X_test_t = torch.tensor(X_test, device=device)
y_test_t = torch.tensor(y_test, device=device)

# Initializing model, loss function (CrossEntropy), and Adam optimizer
model = QNeuralClassifier(input_dim=X_train.shape[2], reservoir_dim=64, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print("\n--- Training on CPU ---")
model.train()
epoch_losses = []
for epoch in range(15):
    optimizer.zero_grad()  # Resetting gradients
    outputs = model(X_train_t)  # Forward pass
    loss = criterion(outputs, y_train_t)  # Computing loss
    loss.backward()  # Backpropagation
    optimizer.step()  # Updating optimizer parameters
    epoch_losses.append(loss.item())
    print(f"Epoch [{epoch+1}/15] | Loss: {loss.item():.4f}")

# Evaluation & Advanced Clinical Metrics Computation
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    preds = torch.max(test_outputs, 1)[1]
    
    # Native performance metrics calculation for binary classification
    tp = ((preds == 1) & (y_test_t == 1)).sum().item()
    fp = ((preds == 1) & (y_test_t == 0)).sum().item()
    fn = ((preds == 0) & (y_test_t == 1)).sum().item()
    tn = ((preds == 0) & (y_test_t == 0)).sum().item()
    
    accuracy = (tp + tn) / len(y_test_t)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n--- Evaluation Results ---")
print(f"Test Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision:      {precision:.4f}")
print(f"Recall (Sens.): {recall:.4f}")
print(f"F1-Score:       {f1_score:.4f}")

In [ ]:
# Step 4: Visualization Module (Loss Curve Plotting)
# Plots the optimization trajectory and training convergence over epochs using Matplotlib.
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o', color='b', linestyle='-')
plt.title('Training Loss Curve - Q-Neural-Dynamics (Amir Kabirian)')
plt.xlabel('Epoch')
plt.ylabel('Cross Entropy Loss')
plt.grid(True)
plt.show()